# CMIP6 IITM-ESM precipitation dataset-source path error

When: 2026-08-26

This notebook reproduces a reported CDS WPS request that subsets two IITM-ESM SSP2-4.5 precipitation (`pr`) collections over 2015–2100. The input combines monthly (`Amon`) and daily (`day`) data in one workflow input.

The service returns: `Process error: A dataset source requires at least one path.`

The notebook preserves the request and provides a guarded reproduction without downloading result data.

## Original WPS workflow

The decoded JSON below is equivalent to the JSON contained in the WPS `ComplexData` CDATA payload. The year list is assembled programmatically for readability.

In [ ]:
collection = [
    (
        "c3s-cmip6.ScenarioMIP.CCCR-IITM.IITM-ESM."
        "ssp245.r1i1p1f1.Amon.pr.gn.v20200915"
    ),
    (
        "c3s-cmip6.ScenarioMIP.CCCR-IITM.IITM-ESM."
        "ssp245.r1i1p1f1.day.pr.gn.v20200915"
    ),
]

months = "jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec"
years = ",".join(str(year) for year in range(2015, 2101))
time_components = f"month:{months}|year:{years}"

request = {
    "inputs": {"pr": collection},
    "steps": {
        "subset_pr_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/pr",
                "area": "-85,-60,-50,20",
                "time_components": time_components,
                "time": "2015/2100",
            },
        }
    },
    "outputs": {"output": "subset_pr_1/output"},
    "doc": "workflow",
}

assert len(collection) == 2
assert years.startswith("2015,2016") and years.endswith("2099,2100")
request

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.

In [ ]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops

In [ ]:
pr = ops.Input("pr", request["inputs"]["pr"])
subset = ops.Subset(
    pr,
    area=request["steps"]["subset_pr_1"]["in"]["area"],
    time=request["steps"]["subset_pr_1"]["in"]["time"],
    time_components=(
        request["steps"]["subset_pr_1"]["in"]["time_components"]
    ),
)

serialized_request = json.loads(subset._serialise())
assert serialized_request == request
serialized_request

## Reproduce the reported error

The next cell submits the full mixed-frequency request. It is disabled by default because the 86-year daily component may make the operation expensive. Set `RUN_REQUEST` to `True` only against the deployment being tested.

In [ ]:
from time import perf_counter

RUN_REQUEST = False

if RUN_REQUEST:
    started_at = perf_counter()
    resp = subset.orchestrate()
    elapsed_seconds = perf_counter() - started_at
    print(f"Orchestration time: {elapsed_seconds:.1f} seconds")
    print(f"Succeeded: {resp.ok}")
    print(f"Status: {resp.status}")
    display(resp)
else:
    print("Request not submitted; set RUN_REQUEST = True to reproduce it.")

## Observed failure

The WPS response contains the following exception:

```xml
<ows:ExceptionText>Process error: A dataset source requires at least one path.</ows:ExceptionText>
```

This indicates that at least one dataset source was constructed without any resolved file paths. The exception does not identify whether the monthly collection, the daily collection, or their combination produced the empty source. Running the two diagnostic workflows below isolates the failing catalog entry before investigating its catalog mapping or available files.

## Isolate the mixed-frequency input

If the combined request fails, submit each collection separately with the same subset parameters. This distinguishes a problem in either source collection from a problem caused by combining monthly and daily time axes. These diagnostic requests are also disabled by default.

In [ ]:
RUN_DIAGNOSTICS = False

diagnostic_workflows = {}
for dataset_id in collection:
    frequency = dataset_id.split(".")[6]
    diagnostic_input = ops.Input("pr", [dataset_id])
    diagnostic_workflows[frequency] = ops.Subset(
        diagnostic_input,
        area=request["steps"]["subset_pr_1"]["in"]["area"],
        time=request["steps"]["subset_pr_1"]["in"]["time"],
        time_components=time_components,
    )

if RUN_DIAGNOSTICS:
    for frequency, workflow in diagnostic_workflows.items():
        diagnostic_response = workflow.orchestrate()
        print(f"{frequency}: {diagnostic_response.ok}, {diagnostic_response.status}")
        display(diagnostic_response)
else:
    print("Diagnostic requests not submitted; set RUN_DIAGNOSTICS = True.")

## Inspect a successful response without loading data

If the combined workflow succeeds, inspect the Metalink size and output URLs without downloading or opening the NetCDF results.

In [ ]:
if RUN_REQUEST and resp.ok:
    print(
        f"Total output size from Metalink: {resp.size:,} bytes "
        f"({resp.size_in_mb:.2f} MiB / {resp.size_in_gb:.3f} GiB)"
    )
    print("\nOutput URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)